In [1]:
# =================================================================================================
# NHANES 2011-2012
# Sleep Disorders Questionnaire
#
# File:
#   SLQ_G.xpt
#
# Purpose:
#   Convert NHANES SLQ_G XPT -> fidelity-preserving corrected CSV.
#
#
# ATTACHED SOURCE FINDINGS:
#
#   Rows                         = 6,175
#   Columns                      = 4
#
#   SEQN range                   = 62161-71916
#   Missing SEQN                 = 0
#   Duplicate SEQN               = 0
#
#   SAS XPORT tiny-zero artifact = 0
#   True numeric zeros           = 0
#
#   Genuine decimal variables    = NONE
#
#
# Variables:
#
#   SEQN
#   SLD010H
#   SLQ050
#   SLQ060
#
#
# Fidelity rules:
#
#   - Remove unnecessary ".0" from SEQN
#
#   - Do NOT replace any values with zero
#
#   - Do NOT round values
#
#   - Do NOT use float_format=
#
#   - Preserve missing-value positions exactly
#
#   - Convert only proven whole-number/code variables to nullable Int64
#
#   - Validate official CDC frequencies
#
#   - Validate XPT -> CSV numeric fidelity
#
#
# GENERAL NHANES RULE:
#
#   Dietary / Laboratory / Examination variables containing genuine
#   decimals must remain AS IS.
#
#   This particular SLQ_G file contains NO genuine decimal variables.
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Questionnnair"
)


XPT_FILE = (
    DATA_DIR
    / "SLQ_G.xpt"
)


CSV_FILE = (
    DATA_DIR
    / "SLQ_G_corrected.csv"
)


# =================================================================================================
# 3. READ ORIGINAL XPT
# =================================================================================================

df = pd.read_sas(
    XPT_FILE,
    format="xport",
    encoding="latin1"
)


#
# Preserve untouched pandas-decoded source.
#
# This is used later to prove that no scientific value
# was altered during cleaning/export.
#

source_df = df.copy()


print("=" * 110)

print(
    "NHANES 2011-2012 SLQ_G XPT -> CSV"
)

print("=" * 110)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Column names:",
    df.columns.tolist()
)


# =================================================================================================
# 4. EXPECTED OFFICIAL STRUCTURE
# =================================================================================================

EXPECTED_COLUMNS = [

    "SEQN",

    "SLD010H",

    "SLQ050",

    "SLQ060"

]


EXPECTED_ROWS = 6175


if (
    df.columns.tolist()
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "SLQ_G column structure differs from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected SLQ_G row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(df):,}"
    )


print(
    "PASS: Structure = 6,175 rows x 4 columns."
)


# =================================================================================================
# 5. VERIFY NUMERIC XPT LOAD
# =================================================================================================

numeric_columns = (

    df.select_dtypes(
        include=[np.number]
    )

    .columns

    .tolist()

)


if (
    numeric_columns
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "Expected all four SLQ_G variables "
        "to load numerically from XPT."
    )


print(
    "PASS: All 4 variables loaded numerically."
)


# =================================================================================================
# 6. KNOWN SAS XPORT TINY-ZERO CHECK
# =================================================================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (

    df[EXPECTED_COLUMNS]

    .eq(TINY_VALUE)

    .sum()

)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- XPORT TINY-ZERO CHECK ---"
)


print(
    tiny_counts.to_string()
)


print(
    "\nTotal tiny XPORT values:",
    tiny_total
)


#
# IMPORTANT:
#
# Attached SLQ_G contains ZERO tiny artifacts.
#
# Therefore we DO NOT perform any replacement.
#

if tiny_total != 0:

    raise ValueError(
        "STOP: Unexpected XPORT tiny-zero values were found.\n\n"
        "The attached validated SLQ_G contained zero tiny artifacts.\n"
        "Do not automatically convert these values without investigating."
    )


print(
    "PASS: No SAS XPORT tiny-zero artifacts."
)


# =================================================================================================
# 7. TRUE NUMERIC ZERO CHECK
# =================================================================================================

zero_counts = (

    df[EXPECTED_COLUMNS]

    .eq(0)

    .sum()

)


zero_total = int(
    zero_counts.sum()
)


print(
    "\n--- TRUE ZERO CHECK ---"
)


print(
    zero_counts.to_string()
)


print(
    "\nTotal numeric zeros:",
    zero_total
)


if zero_total != 0:

    raise ValueError(
        "Unexpected true numeric zeros found in SLQ_G."
    )


print(
    "PASS: No zero-coded observations exist "
    "in this SLQ_G source."
)


# =================================================================================================
# 8. FRACTIONAL VALUE DISCOVERY
# =================================================================================================

#
# IMPORTANT:
#
# np.round() is used ONLY as a diagnostic comparison.
#
# It does NOT modify any actual value.
#

print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)


fractional_counts = {}


for col in EXPECTED_COLUMNS:

    values = (

        df[col]

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    if len(values) == 0:

        fractional_count = 0


    else:

        fractional_count = int(

            (
                ~np.isclose(

                    values,

                    np.round(
                        values
                    ),

                    rtol=0,

                    atol=1e-12

                )
            )

            .sum()

        )


    fractional_counts[
        col
    ] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional values"
    )


fractional_variables = {

    col: count

    for col, count in fractional_counts.items()

    if count > 0

}


if fractional_variables:

    raise ValueError(
        "Unexpected genuine fractional values found:\n"
        + str(
            fractional_variables
        )
    )


print(
    "PASS: No genuine decimal variables exist in SLQ_G."
)


# =================================================================================================
# 9. FIELD TYPE PLAN
# =================================================================================================

INTEGER_COLUMNS = [

    "SEQN",

    "SLD010H",

    "SLQ050",

    "SLQ060"

]


DECIMAL_COLUMNS = []


print(
    "\n--- FIELD TYPE PLAN ---"
)


print(
    "Whole-number/code variables:",
    INTEGER_COLUMNS
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


# =================================================================================================
# 10. SEQN SOURCE VALIDATION
# =================================================================================================

print(
    "\n--- SEQN SOURCE CHECK ---"
)


seqn_missing = int(

    df["SEQN"]
    .isna()
    .sum()

)


seqn_duplicate = int(

    df["SEQN"]
    .duplicated()
    .sum()

)


seqn_min = (
    df["SEQN"]
    .min()
)


seqn_max = (
    df["SEQN"]
    .max()
)


print(
    "Minimum:",
    seqn_min
)


print(
    "Maximum:",
    seqn_max
)


print(
    "Missing:",
    seqn_missing
)


print(
    "Duplicates:",
    seqn_duplicate
)


if seqn_min != 62161:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if seqn_max != 71916:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if seqn_missing != 0:

    raise ValueError(
        "SEQN contains missing observations."
    )


if seqn_duplicate != 0:

    raise ValueError(
        "SEQN contains duplicate observations."
    )


print(
    "PASS: SEQN 62161-71916 validated."
)


# =================================================================================================
# 11. CDC CHECK - SLD010H
# =================================================================================================

#
# Official CDC 2011-2012 frequencies:
#
# 2-11 hours        = 6,123
# 12 hours or more  =    35
# 77 Refused        =     1
# 99 Don't know     =     7
# Missing           =     9
#
# Total             = 6,175
#

sld010h_2_to_11 = int(

    df["SLD010H"]

    .between(
        2,
        11
    )

    .sum()

)


sld010h_12 = int(

    df["SLD010H"]

    .eq(12)

    .sum()

)


sld010h_77 = int(

    df["SLD010H"]

    .eq(77)

    .sum()

)


sld010h_99 = int(

    df["SLD010H"]

    .eq(99)

    .sum()

)


sld010h_missing = int(

    df["SLD010H"]

    .isna()

    .sum()

)


print(
    "\n--- CDC CHECK: SLD010H ---"
)


print(
    "2-11 hours:",
    f"{sld010h_2_to_11:,}"
)


print(
    "12 hours or more:",
    sld010h_12
)


print(
    "77 Refused:",
    sld010h_77
)


print(
    "99 Don't know:",
    sld010h_99
)


print(
    "Missing:",
    sld010h_missing
)


if sld010h_2_to_11 != 6123:

    raise ValueError(
        "SLD010H 2-11 count differs from CDC."
    )


if sld010h_12 != 35:

    raise ValueError(
        "SLD010H=12 count differs from CDC."
    )


if sld010h_77 != 1:

    raise ValueError(
        "SLD010H=77 Refused count differs from CDC."
    )


if sld010h_99 != 7:

    raise ValueError(
        "SLD010H=99 Don't know count differs from CDC."
    )


if sld010h_missing != 9:

    raise ValueError(
        "SLD010H missing count differs from CDC."
    )


print(
    "PASS: SLD010H matches CDC exactly."
)


# =================================================================================================
# 12. CDC CHECK - SLQ050
# =================================================================================================

#
# Official CDC:
#
# 1 = Yes              1,380
# 2 = No               4,792
# 7 = Refused              2
# 9 = Don't know           0
# Missing                  1
#

SLQ050_EXPECTED = {

    1.0: 1380,

    2.0: 4792,

    7.0: 2

}


slq050_actual = (

    df["SLQ050"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


slq050_missing = int(

    df["SLQ050"]

    .isna()

    .sum()

)


print(
    "\n--- CDC CHECK: SLQ050 ---"
)


print(
    "Observed:",
    slq050_actual
)


print(
    "Missing:",
    slq050_missing
)


if slq050_actual != SLQ050_EXPECTED:

    raise ValueError(
        "SLQ050 distribution differs from CDC.\n\n"
        f"Expected:\n{SLQ050_EXPECTED}\n\n"
        f"Actual:\n{slq050_actual}"
    )


if slq050_missing != 1:

    raise ValueError(
        "SLQ050 missing count differs from CDC."
    )


print(
    "PASS: SLQ050 matches CDC exactly."
)


# =================================================================================================
# 13. CDC CHECK - SLQ060
# =================================================================================================

#
# Official CDC:
#
# 1 = Yes                485
# 2 = No               5,682
# 7 = Refused              2
# 9 = Don't know           5
# Missing                  1
#

SLQ060_EXPECTED = {

    1.0: 485,

    2.0: 5682,

    7.0: 2,

    9.0: 5

}


slq060_actual = (

    df["SLQ060"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


slq060_missing = int(

    df["SLQ060"]

    .isna()

    .sum()

)


print(
    "\n--- CDC CHECK: SLQ060 ---"
)


print(
    "Observed:",
    slq060_actual
)


print(
    "Missing:",
    slq060_missing
)


if slq060_actual != SLQ060_EXPECTED:

    raise ValueError(
        "SLQ060 distribution differs from CDC.\n\n"
        f"Expected:\n{SLQ060_EXPECTED}\n\n"
        f"Actual:\n{slq060_actual}"
    )


if slq060_missing != 1:

    raise ValueError(
        "SLQ060 missing count differs from CDC."
    )


print(
    "PASS: SLQ060 matches CDC exactly."
)


# =================================================================================================
# 14. VERIFY SOURCE DATA HAVE NOT BEEN CHANGED
# =================================================================================================

#
# No tiny-zero restoration is necessary in SLQ_G.
#
# Therefore current numeric values must still be exactly the
# same as the pandas-decoded source XPT.
#

source_value_failures = []


for col in EXPECTED_COLUMNS:

    source_values = (

        source_df[col]

        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        df[col]

        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        source_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        source_value_failures.append(
            col
        )


if source_value_failures:

    raise ValueError(
        "Unexpected source values changed:\n"
        + str(
            source_value_failures
        )
    )


print(
    "PASS: No scientific/source values changed."
)


# =================================================================================================
# 15. MISSING-POSITION FIDELITY
# =================================================================================================

missing_position_errors = 0


for col in EXPECTED_COLUMNS:

    source_missing = (

        source_df[col]

        .isna()

        .to_numpy()

    )


    current_missing = (

        df[col]

        .isna()

        .to_numpy()

    )


    missing_position_errors += int(

        (
            source_missing
            !=
            current_missing
        )

        .sum()

    )


if missing_position_errors != 0:

    raise ValueError(
        "Missing-value positions changed during cleaning."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# =================================================================================================
# 16. CONVERT PROVEN WHOLE-NUMBER FIELDS TO NULLABLE Int64
# =================================================================================================

#
# This changes CSV formatting such as:
#
#     62161.0
#
# to:
#
#     62161
#
#
# and:
#
#     1.0
#
# to:
#
#     1
#
#
# ONLY because every nonmissing value in these fields was
# explicitly verified to be mathematically integral.
#

for col in INTEGER_COLUMNS:

    values = (

        df[col]

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    if len(values) > 0:

        integral_mask = np.isclose(

            values,

            np.round(
                values
            ),

            rtol=0,

            atol=1e-12

        )


        if not integral_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    df[col] = (

        df[col]

        .astype("Int64")

    )


print(
    "PASS: Proven whole-number/code fields converted to Int64."
)


# =================================================================================================
# 17. PRE-EXPORT NUMERIC FIDELITY CHECK
# =================================================================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    source_values = (

        source_df[col]

        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        pd.to_numeric(
            df[col],
            errors="coerce"
        )

        .to_numpy(
            dtype=float,
            na_value=np.nan
        )

    )


    same = np.allclose(

        source_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Pre-export numeric fidelity failures:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export numeric fidelity."
)


# =================================================================================================
# 18. EXPORT CORRECTED CSV
# =================================================================================================

#
# IMPORTANT:
#
# Do NOT use:
#
#     float_format=
#
# Do NOT round.
#
# This is especially important across the broader NHANES
# pipeline for Dietary / Lab / Examination decimal values.
#

df.to_csv(

    CSV_FILE,

    index=False,

    na_rep=""

)


print(
    "\nCSV created:"
)


print(
    CSV_FILE
)


# =================================================================================================
# 19. RE-READ CSV WITH ROUND-TRIP NUMERIC PARSER
# =================================================================================================

roundtrip_df = pd.read_csv(

    CSV_FILE,

    low_memory=False,

    float_precision="round_trip"

)


if roundtrip_df.shape != (

    EXPECTED_ROWS,

    len(EXPECTED_COLUMNS)

):

    raise ValueError(
        "Final CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "Final CSV column order changed."
    )


print(
    "PASS: Final CSV structure preserved."
)


# =================================================================================================
# 20. EXACT NUMERIC ROUND-TRIP FIDELITY
# =================================================================================================

roundtrip_failures = []


for col in EXPECTED_COLUMNS:

    source_values = (

        source_df[col]

        .to_numpy(
            dtype=float
        )

    )


    final_values = (

        roundtrip_df[col]

        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        source_values,

        final_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        roundtrip_failures.append(
            col
        )


if roundtrip_failures:

    raise ValueError(
        "XPT -> CSV numeric fidelity failures:\n"
        + str(
            roundtrip_failures
        )
    )


print(
    "PASS: All 4 variables round-trip exactly."
)


# =================================================================================================
# 21. READ FINAL CSV AS RAW TEXT
# =================================================================================================

raw_csv = pd.read_csv(

    CSV_FILE,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 22. UNWANTED ".0" CHECK
# =================================================================================================

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    bad_mask = (

        raw_csv[col]
        .ne("")

        &

        raw_csv[col]
        .str.endswith(".0")

    )


    bad_count = int(
        bad_mask.sum()
    )


    if bad_count > 0:

        dot_zero_errors[
            col
        ] = bad_count


if dot_zero_errors:

    raise ValueError(
        "Unnecessary '.0' formatting remains:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' formatting."
)


# =================================================================================================
# 23. FINAL SEQN TEXT CHECK
# =================================================================================================

if (

    raw_csv["SEQN"]
    .eq("")
    .any()

):

    raise ValueError(
        "Final CSV contains blank SEQN."
    )


if (

    raw_csv["SEQN"]
    .duplicated()
    .any()

):

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (

    raw_csv["SEQN"]
    .str.endswith(".0")
    .any()

):

    raise ValueError(
        "Final CSV SEQN contains '.0'."
    )


if (

    ~raw_csv["SEQN"]
    .str.fullmatch(r"\d+")

).any():

    raise ValueError(
        "Final CSV contains invalid SEQN formatting."
    )


seqn_numeric = pd.to_numeric(

    raw_csv["SEQN"],

    errors="raise"

)


if seqn_numeric.min() != 62161:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if seqn_numeric.max() != 71916:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


print(
    "PASS: Final SEQN is clean."
)


# =================================================================================================
# 24. FINAL TINY-XPORT CHECK
# =================================================================================================

final_tiny_total = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    final_tiny_total += int(

        values
        .eq(TINY_VALUE)
        .sum()

    )


if final_tiny_total != 0:

    raise ValueError(
        "Tiny XPORT artifacts exist in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts in final CSV."
)


# =================================================================================================
# 25. FINAL TRUE-ZERO CHECK
# =================================================================================================

final_zero_total = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    final_zero_total += int(

        values
        .eq(0)
        .sum()

    )


if final_zero_total != 0:

    raise ValueError(
        "Unexpected zero values exist in final CSV."
    )


print(
    "PASS: Final CSV contains no unexpected true zeros."
)


# =================================================================================================
# 26. SCIENTIFIC NOTATION CHECK
# =================================================================================================

scientific_notation_errors = {}


for col in EXPECTED_COLUMNS:

    count = int(

        raw_csv[col]

        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )

        .sum()

    )


    if count > 0:

        scientific_notation_errors[
            col
        ] = count


if scientific_notation_errors:

    raise ValueError(
        "Unexpected scientific notation found:\n"
        + str(
            scientific_notation_errors
        )
    )


print(
    "PASS: No scientific notation."
)


# =================================================================================================
# 27. FINAL MISSING-POSITION CHECK
# =================================================================================================

final_missing_position_errors = 0


for col in EXPECTED_COLUMNS:

    source_missing = (

        source_df[col]

        .isna()

        .to_numpy()

    )


    csv_missing = (

        raw_csv[col]

        .eq("")

        .to_numpy()

    )


    final_missing_position_errors += int(

        (
            source_missing
            !=
            csv_missing
        )

        .sum()

    )


if final_missing_position_errors != 0:

    raise ValueError(
        "Final CSV missing positions differ from XPT."
    )


print(
    "PASS: Missing positions preserved through CSV export."
)


# =================================================================================================
# 28. FINAL CDC FREQUENCY RECHECK
# =================================================================================================

final_numeric = pd.read_csv(

    CSV_FILE,

    low_memory=False,

    float_precision="round_trip"

)


# -------------------------------------------------------------------------------------------------
# SLD010H
# -------------------------------------------------------------------------------------------------

if int(

    final_numeric["SLD010H"]

    .between(
        2,
        11
    )

    .sum()

) != 6123:

    raise ValueError(
        "Final SLD010H 2-11 count differs."
    )


if int(

    final_numeric["SLD010H"]

    .eq(12)

    .sum()

) != 35:

    raise ValueError(
        "Final SLD010H=12 count differs."
    )


if int(

    final_numeric["SLD010H"]

    .eq(77)

    .sum()

) != 1:

    raise ValueError(
        "Final SLD010H=77 count differs."
    )


if int(

    final_numeric["SLD010H"]

    .eq(99)

    .sum()

) != 7:

    raise ValueError(
        "Final SLD010H=99 count differs."
    )


if int(

    final_numeric["SLD010H"]

    .isna()

    .sum()

) != 9:

    raise ValueError(
        "Final SLD010H missing count differs."
    )


# -------------------------------------------------------------------------------------------------
# SLQ050
# -------------------------------------------------------------------------------------------------

final_slq050 = (

    final_numeric["SLQ050"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


if final_slq050 != {

    1: 1380,

    2: 4792,

    7: 2

}:

    raise ValueError(
        "Final SLQ050 distribution differs."
    )


if int(

    final_numeric["SLQ050"]

    .isna()

    .sum()

) != 1:

    raise ValueError(
        "Final SLQ050 missing count differs."
    )


# -------------------------------------------------------------------------------------------------
# SLQ060
# -------------------------------------------------------------------------------------------------

final_slq060 = (

    final_numeric["SLQ060"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


if final_slq060 != {

    1: 485,

    2: 5682,

    7: 2,

    9: 5

}:

    raise ValueError(
        "Final SLQ060 distribution differs."
    )


if int(

    final_numeric["SLQ060"]

    .isna()

    .sum()

) != 1:

    raise ValueError(
        "Final SLQ060 missing count differs."
    )


print(
    "PASS: Final CSV frequencies match CDC."
)


# =================================================================================================
# 29. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2011-2012 SLQ_G CONVERSION COMPLETE"
)


print(
    "=" * 110
)


print(
    "Cycle:"
)


print(
    "  2011-2012"
)


print(
    ""
)


print(
    "Component:"
)


print(
    "  Questionnaire - Sleep Disorders"
)


print(
    ""
)


print(
    "Eligible population:"
)


print(
    "  Age 16 years and older"
)


print(
    ""
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    ""
)


print(
    "Fields:"
)


print(
    "  SEQN"
)


print(
    "  SLD010H"
)


print(
    "  SLQ050"
)


print(
    "  SLQ060"
)


print(
    ""
)


print(
    "Original tiny XPORT artifacts:",
    tiny_total
)


print(
    "Validated zero restorations:",
    0
)


print(
    "True zero values:",
    0
)


print(
    ""
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    ""
)


print(
    "SEQN '.0': 0"
)


print(
    "Duplicate SEQN: 0"
)


print(
    "Unexpected source-value changes: 0"
)


print(
    "Missing-position changes: 0"
)


print(
    "Scientific notation: 0"
)


print(
    "Round-trip numeric failures: 0"
)


print(
    ""
)


print(
    "CDC validation:"
)


print(
    "  SLD010H: PASS"
)


print(
    "  SLQ050:  PASS"
)


print(
    "  SLQ060:  PASS"
)


print(
    "=" * 110
)

NHANES 2011-2012 SLQ_G XPT -> CSV
Rows: 6,175
Columns: 4
Column names: ['SEQN', 'SLD010H', 'SLQ050', 'SLQ060']
PASS: Structure = 6,175 rows x 4 columns.
PASS: All 4 variables loaded numerically.

--- XPORT TINY-ZERO CHECK ---
SEQN       0
SLD010H    0
SLQ050     0
SLQ060     0

Total tiny XPORT values: 0
PASS: No SAS XPORT tiny-zero artifacts.

--- TRUE ZERO CHECK ---
SEQN       0
SLD010H    0
SLQ050     0
SLQ060     0

Total numeric zeros: 0
PASS: No zero-coded observations exist in this SLQ_G source.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional values
SLD010H: 0 fractional values
SLQ050: 0 fractional values
SLQ060: 0 fractional values
PASS: No genuine decimal variables exist in SLQ_G.

--- FIELD TYPE PLAN ---
Whole-number/code variables: ['SEQN', 'SLD010H', 'SLQ050', 'SLQ060']
Genuine decimal variables: []

--- SEQN SOURCE CHECK ---
Minimum: 62161.0
Maximum: 71916.0
Missing: 0
Duplicates: 0
PASS: SEQN 62161-71916 validated.

--- CDC CHECK: SLD010H ---
2-11 hours: 6,123
12 hours